## Суть решения (коротко)

1. **Данные.** Из `events.csv.gz` оставляем только события внутри окна наблюдения каждой cookie (`window_start_ts <= event_ts < window_end_ts`), убираем точные дубли строк, нормализуем `platform` (11 написаний → `web / android / ios`).

Были добавлены новые признаки:
2. **Признаки** — «поведенческий отпечаток» cookie за окно, собранный только по её событиям:
   * **тайминги** — самая сильная группа: медиана/квантили/CV интервалов между событиями, доля интервалов < 1 с, максимум событий в минуту/10 с, автокорреляция интервалов, энтропия по часам. Робот отличается от человека моментами времени (ритм, регулярность);
   * **разнообразие просмотренного**: число уникальных объявлений / категорий / локаций / запросов, максимальная страница выдачи, просмотров на объявление;
   * **состав событий**: счётчики и доли 10 типов событий, отношения «контакты / просмотры», «избранное / просмотры», капча, логин;
   * **курсор** (`pointer_x/y`): доля событий с координатами, доля `(0, 0)`, разброс, длина «прыжков», доля движений строго по одной оси;
   * **User-Agent**: флаги библиотечных клиентов (`python-requests`, `Go-http-client`, `okhttp`…), браузер, мажорная версия, ОС, несоответствие `platform` ↔ UA. Сырой UA как категорию не берём (слишком много уникальных строк);
   * **возраст cookie** на начало и конец окна;
   * **межкукная синхронность** (`sync_*`, `item_sync_*`): сколько чужих событий было рядом по времени / по тому же объявлению — только назад во времени (см. ниже);
   * **композиты** (интенсивность × регулярность × однообразие и др.).
3. **Пропуски** (нет событий, нет курсора и т.д.): счётчики/доли → `0`; величины, которые без событий не определены (интервалы, разброс курсора, энтропия) → `-1`, чтобы деревья отделяли «нет данных» от реальных значений; категории → `"unknown"`. Пропуски в исходных полях событий (`item_id`, `pointer_*`, `search_*`) не заполняются: их наличие само является сигналом (`pointer_present_ratio`, `cnt_*`).
4. **Модель.** Ансамбль **LightGBM + XGBoost** (взвешенное среднее вероятностей, вес подбирается на валидации, в последнем прогоне ≈ 0.6 на LightGBM), `class_weight="balanced"` через `sample_weight` (ботов ≈ 8%). Гиперпараметры подобраны Optuna (TPE, seed 42, 130 trials на каждую модель) по средней метрике P@R≥0.7 на временных фолдах. Итоговые модели переобучаются на **всём** train.

## Как гарантируется «признаки доступны на момент конца окна»

* Все признаки одной cookie считаются только по событиям внутри её окна (события после `window_end_ts` отбрасываются).
* Межкукные признаки считаются **причинно**: для события в момент `t` учитываются только чужие события из `(t − W, t]`, пул общий для train и test (без `target`), плюс поправка на плотность пула. Никаких признаков, зависящих от состава выборки (`cohort_size` удалён), и никаких target-encoding.
* Автотест `check_point_in_time`: если выбросить все события после времени `T`, признаки всех cookie с `window_end <= T` не должны измениться (ячейка с межкукными признаками).

## Валидация

* Тест лежит позже train по времени, поэтому **holdout по времени**: последние 20% train по `window_start_ts` (порог `2026-04-16 00:00`).
* **Зазор (purge):** в обучение попадают только cookie, чьё окно *закончилось* до начала валидации, — окна не перекрываются.
* Для стабильности и подбора гиперпараметров — **3 временных фолда** (expanding window, с тем же зазором). Оценка через тот же `metric.py`, что и на платформе.

## Что сравнивалось

| Модель | P@R≥0.7 (holdout) |
|---|---|
| Константа (доля ботов в holdout) | ≈ 0.08 |
| Baseline: RandomForest на 2 признаках (`n_events`, `item_nunique`) | ≈ 0.09 |
| LightGBM, все признаки (ручные параметры) | ≈ 0.69 |
| XGBoost, все признаки (ручные параметры) | ≈ 0.66 |
| LightGBM + Optuna | ≈ 0.73 |
| XGBoost + Optuna | ≈ 0.73 |
| Ансамбль LGBM + XGB | ≈ 0.74–0.75 |


**Абляции** (LightGBM, ячейка «Ablation»): без timing-признаков метрика падает сильнее всего; без композитов, без возраста cookie и без межкукных `sync_*` — заметно (порядка 1–4 п.п.); `item_sync_*`, `bot_ua_flag`, `app_lib_ua_flag` вклада не дали (в пуле слишком мало пересечений по объявлениям; сигнал UA уже покрыт `multi_signal_bot`).


## Воспроизводимость

* **Python 3.13.2**; библиотеки — в `requirements.txt` (первая ячейка печатает фактические версии).
* Все случайности фиксированы: `RANDOM_STATE = 42` (LightGBM, XGBoost, RandomForest, Optuna TPE).
* Запуск: положить рядом `data/train.csv`, `data/test.csv`, `data/events.csv.gz` и `metric.py` → *Restart & Run All*. Результат — `submission.csv` (`cookie_id`, `score`).
* Optuna по умолчанию выключена: найденные параметры сохранены в коде, чтобы прогон был быстрым и детерминированным
* Используются только open-source библиотеки: pandas, NumPy, scikit-learn (RandomForest, метрики, `permutation_importance`), LightGBM, XGBoost, Optuna

In [1]:
# ============================================================================
# МОДУЛЬ 0 — ИМПОРТЫ И ГЛОБАЛЬНЫЕ НАСТРОЙКИ
# ============================================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
DATA_DIR = Path("data")

# Доли train, попадающие в валидацию (по времени)
VALID_QUANTILE = 0.80

EVENT_NAMES = [
    "item_view", "search_results_view", "photo_swipe", "favorite_add",
    "seller_page_view", "contact_phone_show", "captcha_shown", "login",
    "contact_chat_open", "contact_message_sent",
]

# Прямые сигнатуры HTTP-библиотек и headless-клиентов
BOT_UA_RE = re.compile(
    r"(python-requests|urllib3|go-http-client|curl/|scrapy|postmanruntime|"
    r"okhttp|axios|node-fetch|headlesschrome|phantomjs|wget|"
    r"\bbot\b|spider|crawler)",
    re.IGNORECASE,
)

# Подсказки для проверки platform <-> UA
BROWSER_HINT_RE = {
    "web":     re.compile(r"windows|macintosh|x11|linux(?!.*android)", re.I),
    "android": re.compile(r"android", re.I),
    "ios":     re.compile(r"iphone|ipad|cpu os|mac os x.*mobile", re.I),
}

# Парсинг UA
UA_BROWSER_RE = re.compile(
    r"(YaBrowser|HeadlessChrome|Edg|OPR|Chrome|Firefox|Safari)/[\d.]+", re.I)
UA_OS_RE = re.compile(
    r"(Windows NT \d+\.\d+|Mac OS X \d+[._]\d+|"
    r"Android \d+|iPhone OS \d+|Linux)", re.I)
UA_VER_RE = re.compile(r"(?:Chrome|Firefox|Edg|OPR|YaBrowser|Safari)/(\d+)")

z:\projects\dev\jump_into_django\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Загрузка и очистка данных

Точные дубли строк событий удаляются (после удаления `eid`, который дублирует `event_name`),

`platform` встречается в 11 написаниях (`WEB/Web/web/desktop`, `ANDROID/Android/android`, `iphone/IOS/iOS/ios`) — сводим к трём значениям,

**Окно наблюдения:** событие учитывается только если `window_start_ts <= event_ts < window_end_ts` своей cookie

In [2]:
# ============================================================================
# МОДУЛЬ 1 — ЗАГРУЗКА И ПРЕДОБРАБОТКА ДАННЫХ
# ============================================================================

def load_data(data_dir: Path):
    # Важно: здесь задаётся исходная временная граница наблюдения.
    # События вне окна cookie дальше не должны попадать в её признаки.
    train = pd.read_csv(
        data_dir / "train.csv",
        parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
    )
    test = pd.read_csv(
        data_dir / "test.csv",
        parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
    )
    events = pd.read_csv(data_dir / "events.csv.gz", parse_dates=["event_ts"])

    events = events.drop(columns=["eid"], errors="ignore")
    before = len(events)
    events = events.drop_duplicates() # считаем как случайные повторы записей, было проверено, что на улучшение метрик не влияет
    dropped = before - len(events)
    if dropped:
        print(f"[load] удалено точных дублей событий: {dropped}")

    events = events.sort_values(["cookie_id", "event_ts"]).reset_index(drop=True)
    return train, test, events


def normalize_platform(events: pd.DataFrame) -> pd.DataFrame:
    """desktop/WEB/Web/web -> web; ANDROID/Android/android -> android; iphone/IOS/iOS -> ios."""
    low = events["platform"].str.lower()
    events = events.copy()
    events["platform_norm"] = np.select(
        [low.eq("android"), low.isin(["ios", "iphone"])],
        ["android", "ios"],
        default="web",
    )
    return events


def filter_window(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Оставляем только события внутри окна наблюдения конкретной cookie."""
    m = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]],
        on="cookie_id", how="inner",
    )
    mask = (m["event_ts"] >= m["window_start_ts"]) & (m["event_ts"] < m["window_end_ts"])
    return m.loc[mask].drop(columns=["window_start_ts", "window_end_ts"])


train, test, events = load_data(DATA_DIR)
events = normalize_platform(events)
print("shapes:", train.shape, test.shape, events.shape)

ev_tr = filter_window(events, train)
ev_te = filter_window(events, test)
print(f"events in train windows: {len(ev_tr):,}, in test windows: {len(ev_te):,}")

all_events = pd.concat([ev_tr, ev_te], ignore_index=True)
print(f"total events: {len(all_events):,}")

[load] удалено точных дублей событий: 4863
shapes: (11091, 5) (4909, 4) (324042, 14)
events in train windows: 195,484, in test windows: 88,349
total events: 283,833


## 2. Признаки

Все признаки доступны на момент `window_end_ts` считаются только по событиям внутри окна (кроме межкукных — они причинные, см. 2.2).

### 2.1 Метаданные cookie
Возраст cookie на начало и конец окна. Статистик по другим cookie здесь нет

In [3]:
# ============================================================================
# МОДУЛЬ 2.1 — ПРИЗНАКИ МЕТАДАННЫХ COOKIE
# ============================================================================

def add_meta_features(meta: pd.DataFrame) -> pd.DataFrame:
    """
    Признаки только из метаданных САМОЙ cookie (никаких счётчиков по другим cookie:
    cohort_size_* удалены — они зависели от состава выборки, а не от поведения).
    Оба возраста известны на момент window_end.
    """
    m = meta
    return pd.DataFrame({
        "cookie_id": m["cookie_id"].values,
        "cookie_age_hours": ((m["window_start_ts"] - m["cookie_created_at"])
                             .dt.total_seconds() / 3600).clip(lower=0).values,
        "cookie_age_end_hours": ((m["window_end_ts"] - m["cookie_created_at"])
                                 .dt.total_seconds() / 3600).clip(lower=0).values,
        "dow_of_window_start": m["window_start_ts"].dt.dayofweek.values, # ни на что не влияет, можно удалить
    })


meta_feats_tr = add_meta_features(train)
meta_feats_te = add_meta_features(test)
print(meta_feats_tr.describe().round(2).T)


                        count     mean      std   min     25%      50%  \
cookie_age_hours      11091.0  1724.81  1607.83   0.3  314.75  1434.22   
cookie_age_end_hours  11091.0  1748.81  1607.83  24.3  338.75  1458.22   
dow_of_window_start   11091.0     2.93     1.98   0.0    1.00     3.00   

                          75%       max  
cookie_age_hours      2605.27  10782.84  
cookie_age_end_hours  2629.27  10806.84  
dow_of_window_start      5.00      6.00  


### 2.2 Межкукные признаки (синхронность), строго «до конца окна»

Идея: боты одного сервиса ходят синхронно и по одним и тем же объявлениям. Но такую статистику легко посчитать с утечкой из будущего, поэтому правила такие:

1. Один пул событий для train и test (иначе у test счётчики будут завышены).
2. Для события в момент `t` считаются только чужие события до момента `t` — ничего из будущего.
3. Абсолютные счётчики нормируются на плотность пула (`rel_60s`), т.к. плотность события меняется во времени.
4. Автотест `check_point_in_time` проверяет отсутствие утечки: признаки не меняются, если удалить события после `T`.

In [4]:
# ============================================================================
# МОДУЛЬ 2.2 — МЕЖКУКНАЯ СИНХРОННОСТЬ И ПРИЧИННЫЕ ПРИЗНАКИ
# ============================================================================

# =====================================================================
# Межкукные признаки (sync, редкость UA) — ТОЛЬКО с накоплением на момент window_end.
#
# Правила:
#  1. Пул один и тот же для train и test (train+test events, без target) —
#     иначе абсолютные счётчики у test систематически больше.
#  2. Для cookie i используются события пула только с event_ts < window_end_i;
#     в sync — даже только с ts <= времени самого события (скользящее окно назад).
#  3. Считаем доли/отношения к фону, а не абсолютные счётчики (плотность пула
#     растёт в начале периода — это дрейф, а не поведение).
# =====================================================================
BIG = 1 << 33                     # ключ = group*BIG + ts_ms (2^33 мс ≈ 99 суток)
OFFSET_MS = 3 * 24 * 3600 * 1000  # чтобы (t - W) не уходило в минус


def to_ms(ts: pd.Series, t0: pd.Timestamp) -> np.ndarray:
    return ((ts - t0) // pd.Timedelta(milliseconds=1)).to_numpy("int64") + OFFSET_MS


def count_in_range(pool_grp, pool_t, q_grp, q_lo, q_hi):
    """Для каждого запроса: число событий пула с группой q_grp и ts в (q_lo, q_hi]."""
    keys = np.sort(pool_grp.astype("int64") * BIG + pool_t)
    hi = np.searchsorted(keys, q_grp.astype("int64") * BIG + q_hi, side="right")
    lo = np.searchsorted(keys, q_grp.astype("int64") * BIG + q_lo, side="right")
    return hi - lo


def _sync_block(t, grp, cid, windows_s, prefix):
    """Причинный sync: события ДРУГИХ cookie с тем же ключом в (t-W, t]."""
    out = {}
    pair = pd.factorize(grp * (cid.max() + 1) + cid)[0].astype("int64")  # (ключ, cookie)
    for W in windows_s:
        w = W * 1000
        all_n = count_in_range(grp, t, grp, t - w, t)
        own_n = count_in_range(pair, t, pair, t - w, t)   # вычитаем собственные события
        out[f"{prefix}{W}s"] = all_n - own_n
    return out


def build_cross_cookie_features(pool, meta, sync_windows=(1, 60), ua_window_h=24):
    """
    pool: события в окнах train+test (единый пул).
    meta: cookie_id, window_start_ts, window_end_ts (кого считаем).
    """
    pool = pool[["cookie_id", "event_ts", "user_agent", "item_id"]].reset_index(drop=True)
    t0 = min(pool["event_ts"].min(), meta["window_start_ts"].min())
    t = to_ms(pool["event_ts"], t0)
    cid = pd.factorize(pool["cookie_id"])[0].astype("int64")
    zeros = np.zeros(len(pool), dtype="int64")

    # 1. sync по всем событиям (замена sync_* и cohort_*)
    ev = pd.DataFrame({"cookie_id": pool["cookie_id"]})
    for k, v in _sync_block(t, zeros, cid, sync_windows, "o_").items():
        ev[k] = v
    dens_per_s = count_in_range(zeros, t, zeros, t - 3600_000, t) / 3600.0  # фон за 1ч
    ev["rel_60s"] = ev["o_60s"] / (dens_per_s * 60 + 1.0)   # поправка на дрейф плотности

    # 2. sync по одному объявлению (замена item_sync_*)
    has_item = pool["item_id"].notna().to_numpy()
    it = pd.factorize(pool.loc[has_item, "item_id"])[0].astype("int64")
    isync = pd.DataFrame({"cookie_id": pool.loc[has_item, "cookie_id"].to_numpy()})
    for k, v in _sync_block(t[has_item], it, cid[has_item], (60,), "item_o_").items():
        isync[k] = v

    g = ev.groupby("cookie_id")
    f = pd.DataFrame({
        "sync_max_1s": g["o_1s"].max(), "sync_mean_1s": g["o_1s"].mean(),
        "sync_frac_1s": g["o_1s"].apply(lambda s: (s >= 3).mean()),
        "sync_max_60s": g["o_60s"].max(), "sync_mean_60s": g["o_60s"].mean(),
        "sync_rel_max_60s": g["rel_60s"].max(),
    })
    gi = isync.groupby("cookie_id")["item_o_60s"]
    f["item_sync_max"], f["item_sync_mean"] = gi.max(), gi.mean()
    f["item_sync_frac_ge2"] = gi.apply(lambda s: (s >= 2).mean())

    return meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left")


def check_point_in_time(pool, meta, T):
    """Признаки cookie с window_end <= T не должны меняться, если выбросить события >= T."""
    m = meta[meta["window_end_ts"] <= T]
    full = build_cross_cookie_features(pool, m).set_index("cookie_id").sort_index()
    cut = build_cross_cookie_features(pool[pool["event_ts"] < T], m).set_index("cookie_id").sort_index()
    bad = [c for c in full.columns if not np.allclose(full[c], cut[c], equal_nan=True)]
    assert not bad, f"утечка из будущего: {bad}"
    print(f"OK: {len(m)} cookie, {full.shape[1]} признаков не зависят от событий после T")


# Единый пул для train и test
cross_tr = build_cross_cookie_features(all_events, train)
cross_te = build_cross_cookie_features(all_events, test)

# Автотест на утечку из будущего
_all_meta = pd.concat([train, test], ignore_index=True)
check_point_in_time(all_events, _all_meta, T=_all_meta["window_end_ts"].median())

print(cross_tr.describe().round(3).T)


OK: 8400 cookie, 9 признаков не зависят от событий после T
                      count    mean    std  min     25%     50%     75%  \
sync_max_1s         11091.0   1.026  0.705  0.0   1.000   1.000   1.000   
sync_mean_1s        11091.0   0.185  0.175  0.0   0.067   0.167   0.250   
sync_frac_1s        11091.0   0.001  0.012  0.0   0.000   0.000   0.000   
sync_max_60s        11091.0  18.159  6.489  0.0  14.000  18.000  22.000   
sync_mean_60s       11091.0  10.905  3.918  0.0   8.333  10.957  13.299   
sync_rel_max_60s    11091.0   1.562  0.541  0.0   1.228   1.517   1.816   
item_sync_max       10918.0   0.001  0.037  0.0   0.000   0.000   0.000   
item_sync_mean      10918.0   0.000  0.002  0.0   0.000   0.000   0.000   
item_sync_frac_ge2  10918.0   0.000  0.000  0.0   0.000   0.000   0.000   

                       max  
sync_max_1s          4.000  
sync_mean_1s         2.000  
sync_frac_1s         0.500  
sync_max_60s        45.000  
sync_mean_60s       34.000  
sync_rel_max_60s

### 2.3 Состав и разнообразие событий
Счётчики и доли типов событий, число уникальных объявлений/категорий/локаций/запросов, максимальная страница выдачи. Человек переключается по разным категориям и локациям, бот обычно либо обходит всё подряд, либо монотонно одно и то же.

In [5]:
# ============================================================================
# МОДУЛЬ 2.3 — СОСТАВ И РАЗНООБРАЗИЕ СОБЫТИЙ
# ============================================================================

def build_event_features(events_win: pd.DataFrame) -> pd.DataFrame:
    g = events_win.groupby("cookie_id")

    n_events = g.size().rename("n_events")
    first_ts = g["event_ts"].min()
    last_ts = g["event_ts"].max()
    span_h = ((last_ts - first_ts).dt.total_seconds() / 3600).rename("active_span_hours")

    ev_counts = (events_win.groupby(["cookie_id", "event_name"]).size()
                 .unstack(fill_value=0).reindex(columns=EVENT_NAMES, fill_value=0))
    ev_ratios = ev_counts.div(ev_counts.sum(axis=1).replace(0, 1), axis=0)
    ev_ratios.columns = [f"ratio_{c}" for c in ev_ratios.columns]
    ev_counts.columns = [f"cnt_{c}" for c in ev_counts.columns]

    diversity = pd.DataFrame({
        "item_nunique": g["item_id"].nunique(),
        "category_nunique": g["item_category"].nunique(),
        "location_nunique": g["item_location"].nunique(),
        "seller_type_nunique": g["seller_type"].nunique(),
        "query_nunique": g["search_query"].nunique(),
        "max_search_page": g["search_page"].max(),
        "ua_nunique": g["user_agent"].nunique(),
        "platform_nunique": g["platform_norm"].nunique(),
    })

    out = pd.concat([n_events, span_h, ev_counts, ev_ratios, diversity], axis=1)

    out["has_captcha"] = (out["cnt_captcha_shown"] > 0).astype(int)
    out["has_login"] = (out["cnt_login"] > 0).astype(int)

    contact_cols = ["cnt_contact_phone_show", "cnt_contact_chat_open", "cnt_contact_message_sent"]
    out["contact_to_view_ratio"] = out[contact_cols].sum(axis=1) / out["cnt_item_view"].clip(lower=1)
    out["favorite_to_view_ratio"] = out["cnt_favorite_add"] / out["cnt_item_view"].clip(lower=1)
    out["views_per_unique_item"] = out["cnt_item_view"] / out["item_nunique"].replace(0, np.nan)

    out.index.name = "cookie_id"
    return out.reset_index()


ev_feats_tr = build_event_features(ev_tr)
ev_feats_te = build_event_features(ev_te)
print("event feats:", ev_feats_tr.shape[1] - 1, "columns")

event feats: 35 columns


### 2.4 Тайминги
Главная группа: интервалы между соседними событиями (среднее, медиана, квантили, CV, форма распределения), пачки событий за 10 с / минуту, автокорреляция, энтропия по часам суток. У cookie с < 2 событий интервалы не определены → значение `-1`.

In [6]:
# ============================================================================
# МОДУЛЬ 2.4 — ВРЕМЕННЫЕ ПАТТЕРНЫ И ТАЙМИНГИ
# ============================================================================

def _timing_features(ts: pd.Series) -> dict:
    ts = ts.sort_values()
    diffs = ts.diff().dt.total_seconds().dropna()
    if len(diffs) == 0:
        return {
            "mean_interval_s": -1.0, "std_interval_s": -1.0, "cv_interval": -1.0,
            "min_interval_s": -1.0, "max_interval_s": -1.0,
            "median_interval_s": -1.0, "frac_interval_lt1s": 0.0,
            "max_events_per_min": 1.0, "max_events_per_10s": 1.0,
        }
    # Интервалы считаются только между соседними событиями одной cookie.
    # Поэтому регулярность описывает ритм действий, а не просто объём трафика.
    mean_dt = diffs.mean()
    std_dt = diffs.std(ddof=0)
    cv = std_dt / mean_dt if mean_dt > 0 else 0.0
    # Группировка по минутам/10 секундам ловит короткие «пачки» событий,
    # которые средний интервал может сгладить.
    per_min = ts.dt.floor("min").value_counts()
    per_10s = ts.dt.floor("10s").value_counts()
    return {
        "mean_interval_s": mean_dt,
        "std_interval_s": std_dt,
        "cv_interval": cv,
        "min_interval_s": diffs.min(),
        "max_interval_s": diffs.max(),
        "median_interval_s": diffs.median(),
        "frac_interval_lt1s": float((diffs < 1).mean()),
        "max_events_per_min": float(per_min.max()),
        "max_events_per_10s": float(per_10s.max()),
    }


def _interval_dist_features(ts: pd.Series) -> dict:
    """Форма распределения интервалов + децили."""
    ts = ts.sort_values()
    d = ts.diff().dt.total_seconds().dropna()
    if len(d) < 3:
        return {
            "iqr_interval_s": -1.0,
            "p95_p5_ratio": -1.0,
            "log_interval_std": -1.0,
            "frac_interval_near_median": 0.0,
            # новые
            "p10_interval_s": -1.0, "p90_interval_s": -1.0,
            "p90_p10_diff": -1.0,
        }
    p5, p10, p25, p50, p75, p90, p95 = np.percentile(
        d, [5, 10, 25, 50, 75, 90, 95])
    iqr = p75 - p25
    ratio = (p95 + 1e-6) / (p5 + 1e-6)
    log_std = float(np.log1p(d).std())
    near = float((np.abs(d - p50) <= 0.2 * p50).mean())
    return {
        "iqr_interval_s": float(iqr),
        "p95_p5_ratio": float(ratio),
        "log_interval_std": log_std,
        "frac_interval_near_median": near,
        # новые: у бота p10 ≈ p90, у человека p90-p10 огромное
        "p10_interval_s": float(p10),
        "p90_interval_s": float(p90),
        "p90_p10_diff": float(p90 - p10),
    }


def _hour_entropy(ts: pd.Series) -> float:
    if len(ts) < 3:
        return -1.0
    p = ts.dt.hour.value_counts(normalize=True)
    return float(-(p * np.log(p + 1e-12)).sum())


def _autocorr(diffs: pd.Series, lag: int) -> float:
    if len(diffs) < lag + 5:
        return np.nan
    return float(pd.Series(diffs.values).autocorr(lag=lag))


def build_timing_features(events_win: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for cid, sub in events_win.groupby("cookie_id")["event_ts"]:
        sub = sub.sort_values()
        r = {"cookie_id": cid}
        r.update(_timing_features(sub))
        r.update(_interval_dist_features(sub))   # <-- новое
        r["hour_entropy"] = _hour_entropy(sub)
        diffs = sub.diff().dt.total_seconds().dropna()
        r["autocorr_lag1"] = _autocorr(diffs, 1)
        r["autocorr_lag2"] = _autocorr(diffs, 2)
        rows.append(r)
    return pd.DataFrame(rows)


timing_tr = build_timing_features(ev_tr)
timing_te = build_timing_features(ev_te)
print("timing feats:", timing_tr.shape[1] - 1)

timing feats: 19


### 2.5 User-Agent
Сырую строку UA не используем — слишком много уникальных значений и она описывает технику, а не поведение. Вместо этого: флаги библиотечных клиентов (`python-requests`, `Go-http-client`, `okhttp`, headless-браузеры), браузер / мажорная версия / ОС основного UA (мода по событиям cookie), рассинхрон `platform` ↔ UA.

In [7]:
# ============================================================================
# МОДУЛЬ 2.5 — USER-AGENT
# ============================================================================

def extract_ua_features(events_win: pd.DataFrame) -> pd.DataFrame:
    def _agg(sub: pd.DataFrame) -> pd.Series:
        uas = sub["user_agent"].dropna()
        if len(uas) == 0:
            return pd.Series({
                "app_lib_ua_flag": 0,
                "ua_browser": "unknown", 
                "ua_browser_major": -1,
                "ua_os": "unknown", 
                "platform_ua_mismatch": 0
            })

        ua_main = uas.mode().iat[0] if not uas.mode().empty else uas.iloc[0]
        app_flag = int(uas.str.contains(r"avito/[\d.]+.*okhttp", na=False, regex=True).any())

        m_browser = UA_BROWSER_RE.search(ua_main)
        browser = m_browser.group(1) if m_browser else "other"
        m_ver = UA_VER_RE.search(ua_main)
        ver = int(m_ver.group(1)) if m_ver else -1
        m_os = UA_OS_RE.search(ua_main)
        os_name = m_os.group(1).split()[0] if m_os else "other"

        plat_vals = sub["platform_norm"].unique()
        mismatch = 0
        if len(plat_vals) == 1:
            pat = BROWSER_HINT_RE[plat_vals[0]]
            mismatch = int(not uas.str.contains(pat, na=False).any())

        return pd.Series({
            "app_lib_ua_flag": app_flag,
            "ua_browser": browser,
            "ua_browser_major": ver,
            "ua_os": os_name,
            "platform_ua_mismatch": mismatch,
        })

    return (events_win.groupby("cookie_id")
            .apply(_agg, include_groups=False)
            .reset_index())


# Глобальная частота UA заменена причинным main_ua_share_24h (см. build_cross_cookie_features)
ua_feats_tr = extract_ua_features(ev_tr)
ua_feats_te = extract_ua_features(ev_te)
print("ua feats:", ua_feats_tr.shape[1] - 1)
print(ua_feats_tr["ua_browser"].value_counts().head())


ua feats: 5
ua_browser
Chrome            6873
other             2341
Firefox           1399
HeadlessChrome     286
Safari             192
Name: count, dtype: int64


### 2.6 Курсор
`pointer_x/y` заполнены не всегда (пропуск = событие без координат, а не ошибка). Считаем долю событий с координатами, долю точек `(0, 0)`, разброс и длину "прыжков" между соседними точками, долю движений строго по одной оси.

In [8]:
# ============================================================================
# МОДУЛЬ 2.6 — ПОВЕДЕНИЕ КУРСОРА
# ============================================================================

def build_pointer_features(events_win: pd.DataFrame) -> pd.DataFrame:
    e = events_win[["cookie_id", "event_ts", "pointer_x", "pointer_y"]].copy()
    e = e.sort_values(["cookie_id", "event_ts"])
    has = e["pointer_x"].notna() & e["pointer_y"].notna()
    e["has"] = has
    e["is_zero"] = has & (e["pointer_x"] == 0) & (e["pointer_y"] == 0)
    e["dist_from_origin"] = np.sqrt(e["pointer_x"] ** 2 + e["pointer_y"] ** 2)

    g = e.groupby("cookie_id")
    out = pd.DataFrame({
        "pointer_present_ratio": g["has"].mean(),
        "pointer_x_std":  g["pointer_x"].std(ddof=0),
        "pointer_y_std":  g["pointer_y"].std(ddof=0),
        "pointer_x_mean": g["pointer_x"].mean(),
        "pointer_y_mean": g["pointer_y"].mean(),
        "pointer_dist_mean": g["dist_from_origin"].mean(),
        "pointer_dist_std":  g["dist_from_origin"].std(ddof=0),
    })

    # доля нулевых среди непустых
    denom = g["has"].sum().replace(0, np.nan)
    out["pointer_zero_ratio"] = (g["is_zero"].sum() / denom).fillna(0.0)

    # Смотрим последовательность только событий, где обе координаты известны.
    # Пропуск координат сам по себе не считается движением.
    # средние прыжки и доля прямолинейных
    def _jumps(sub: pd.DataFrame) -> pd.Series:
        s = sub[sub["has"]].copy()
        if len(s) < 3:
            return pd.Series({"pointer_jump_mean": -1.0, "pointer_line_ratio": 0.0})
        dx = s["pointer_x"].diff()
        dy = s["pointer_y"].diff()
        dist = np.sqrt(dx * dx + dy * dy).dropna()
        line = ((dx.abs() < 2) ^ (dy.abs() < 2)).astype(float).mean()
        return pd.Series({"pointer_jump_mean": dist.mean(), "pointer_line_ratio": line})

    extra = e.groupby("cookie_id").apply(_jumps).reset_index()
    out = out.reset_index().merge(extra, on="cookie_id", how="left")
    return out


ptr_tr = build_pointer_features(ev_tr)
ptr_te = build_pointer_features(ev_te)
print("pointer feats:", ptr_tr.shape[1] - 1)

C:\Users\098ja\AppData\Local\Temp\ipykernel_19968\336155139.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  extra = e.groupby("cookie_id").apply(_jumps).reset_index()


pointer feats: 10


C:\Users\098ja\AppData\Local\Temp\ipykernel_19968\336155139.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  extra = e.groupby("cookie_id").apply(_jumps).reset_index()


### 2.7 Сборка матрицы признаков, композиты, пропуски и категории

Политика пропусков: счётчики/доли → `0`; неопределённые без событий величины → `-1`; категории → `"unknown"`. Категориальные признаки в test приводятся к тем же категориям, что в train

In [9]:
# ============================================================================
# МОДУЛЬ 2.7 — СБОРКА ПРИЗНАКОВ И КОМПОЗИТНЫЕ МЕТРИКИ
# ============================================================================

def assemble_features(meta, ev_feats, timing, ua, ptr, cross, meta_feats):
    df = meta[["cookie_id"]].copy()
    for piece in (ev_feats, timing, ua, ptr, cross, meta_feats):
        df = df.merge(piece, on="cookie_id", how="left")
    return df



def add_composites(feat: pd.DataFrame) -> pd.DataFrame:
    f = feat.copy()
    f["n_events"] = f["n_events"].fillna(0)

    # log1p сжимает хвост числа событий: разница между 10 и 100 событиями
    # не должна доминировать над остальными поведенческими сигналами.
    intensity = np.log1p(f["n_events"]) / (f["active_span_hours"].clip(lower=1e-3).fillna(1))
    regularity = 1.0 / (1.0 + f["cv_interval"].fillna(5.0).clip(lower=0))
    monotony = 1.0 - (f["item_nunique"].fillna(0) /
                      f["cnt_item_view"].clip(lower=1).fillna(1))
    monotony = monotony.clip(lower=0, upper=1)

    f["intensity"] = intensity
    f["regularity"] = regularity
    f["monotony"] = monotony
    f["irm_combo"] = intensity * regularity * monotony

    f["sync_x_cv"] = f["sync_max_1s"].fillna(0) / (1.0 + f["cv_interval"].fillna(5.0).clip(lower=0))
    f["sync_x_regularity"] = f["sync_max_1s"].fillna(0) * regularity

    f["hour_entropy_per_event"] = f["hour_entropy"].fillna(-1) / np.sqrt(f["n_events"].clip(lower=1))

    # Композит объединяет несколько уже рассчитанных сигналов в один признак.
    f["multi_signal_bot"] = (
        f["app_lib_ua_flag"].fillna(0).astype(int)
        + (f["platform_ua_mismatch"].fillna(0) > 0).astype(int)
    )
    return f


raw_tr = assemble_features(
    train,
    ev_feats_tr,
    timing_tr,
    ua_feats_tr,
    ptr_tr,
    cross_tr,
    meta_feats_tr,
)

raw_te = assemble_features(
    test,
    ev_feats_te,
    timing_te,
    ua_feats_te,
    ptr_te,
    cross_te,
    meta_feats_te,
)

Xtr = add_composites(raw_tr)
Xte = add_composites(raw_te)

# Выравниваем колонки
for c in Xtr.columns:
    if c not in Xte.columns:
        Xte[c] = np.nan
Xte = Xte[Xtr.columns]

# Явные заполнения
zero_cols = [c for c in Xtr.columns if c.startswith(("cnt_", "ratio_"))]
zero_cols += [
    "n_events", "item_nunique", "category_nunique", "location_nunique",
    "seller_type_nunique", "query_nunique", "ua_nunique", "platform_nunique",
    "frac_interval_lt1s", "max_events_per_min", "max_events_per_10s",
    "pointer_present_ratio", "pointer_zero_ratio",
    "contact_to_view_ratio", "favorite_to_view_ratio",
    "has_captcha", "has_login", "bot_ua_flag", "app_lib_ua_flag",
    "platform_ua_mismatch", "multi_signal_bot",
    # межкукные признаки: нет событий -> нет синхронности
    "sync_max_1s", "sync_mean_1s", "sync_frac_1s", "sync_max_60s", "sync_mean_60s",
    "sync_rel_max_60s", "item_sync_max", "item_sync_mean", "item_sync_frac_ge2",
    "main_ua_cnt_24h", "main_ua_share_24h",
]
zero_cols = [c for c in zero_cols if c in Xtr.columns]
Xtr[zero_cols] = Xtr[zero_cols].fillna(0)
Xte[zero_cols] = Xte[zero_cols].fillna(0)

sentinel_cols = [
    "max_search_page", "mean_interval_s", "std_interval_s", "cv_interval",
    "min_interval_s", "max_interval_s", "median_interval_s",
    "hour_entropy", "views_per_unique_item", "autocorr_lag1", "autocorr_lag2",
    "pointer_x_std", "pointer_y_std", "pointer_jump_mean",
    "pointer_x_mean", "pointer_y_mean", "pointer_dist_mean", "pointer_dist_std",
    "hour_entropy_per_event",
    "iqr_interval_s", "p95_p5_ratio", "log_interval_std",
    "p10_interval_s", "p90_interval_s", "p90_p10_diff",
]
sentinel_cols = [c for c in sentinel_cols if c in Xtr.columns]
Xtr[sentinel_cols] = Xtr[sentinel_cols].fillna(-1)
Xte[sentinel_cols] = Xte[sentinel_cols].fillna(-1)

# Категориальные
for c in ["ua_browser", "ua_os", "platform_norm"]:
    if c in Xtr.columns:
        Xtr[c] = Xtr[c].fillna("unknown").astype("category")
        # те же категории, что в train (иначе LightGBM/XGB получат несовместимые коды)
        Xte[c] = pd.Categorical(Xte[c].fillna("unknown"), categories=Xtr[c].cat.categories)

cat_cols = [c for c in ["ua_browser", "ua_os"] if c in Xtr.columns]

feature_cols = [c for c in Xtr.columns if c != "cookie_id"]
DROP = [
    "bot_ua_flag",              # стабильно вредит
    "hour_of_window_start",     # всегда 0
    "window_hours",             # всегда 24

    # # Композиты — вредят (ablate +0.034)
    # "irm_combo", "intensity", "regularity", "monotony",
    # "sync_x_cv", "sync_x_regularity",
    
    # # Sync по всем событиям — вредят (ablate +0.007)
    # "sync_max_sec", "sync_mean_sec", "sync_frac_sec",
    # "sync_max_min", "sync_frac_min",
    
    # # First-event sync — почти нули (mean 0.010, max 1)
    # "cohort_sec", "cohort_min",
    
    # # Item sync — почти нули (mean 0.001)
    # "item_sync_max", "item_sync_mean", "item_sync_frac_ge2",
    
    # # Cohort size — почти константы (std 92 при диапазоне 606-912)
    # "cohort_size_sec", "cohort_size_min",
    
    # # Compоsite, который их использовал
    # "multi_signal_bot",
]
feature_cols = [c for c in feature_cols if c not in DROP]
print(f"total features: {len(feature_cols)}")
print(f"categorical:    {cat_cols}")

y = train["target"].values
print("target distribution:", np.bincount(y))

total features: 89
categorical:    ['ua_browser', 'ua_os']
target distribution: [10192   899]


## 3. Валидация и baseline

Holdout по времени (последние 20% окон) с зазором: в обучение только cookie, чьё окно закончилось до начала валидации. Baseline — константа и RandomForest на двух простых признаках (число событий, число уникальных объявлений).

In [11]:
# ============================================================================
# МОДУЛЬ 3 — ВРЕМЕННАЯ ВАЛИДАЦИЯ И BASELINE
# ============================================================================

from metric import precision_at_recall  # официальная метрика соревнования

valid_threshold_ts = train["window_start_ts"].quantile(VALID_QUANTILE)
is_valid = (train["window_start_ts"] >= valid_threshold_ts).values
# зазор: в обучение только cookie, чьё окно ЗАКОНЧИЛОСЬ до начала валидации
is_train = (train["window_end_ts"] <= valid_threshold_ts).values
print(f"valid threshold: {valid_threshold_ts}, "
      f"train: {is_train.sum()}, valid: {is_valid.sum()}")

y_tr, y_val = y[is_train], y[is_valid]

print("\n=== Baseline ===")
print(f"константа (доля ботов): P@R0.7 = {y_val.mean():.4f}")

simple_cols = ["n_events", "item_nunique"]
rf_simple = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)
rf_simple.fit(Xtr.loc[is_train, simple_cols], y_tr)
p_simple = rf_simple.predict_proba(Xtr.loc[is_valid, simple_cols])[:, 1]
print(f"RF (2 признака):         P@R0.7 = {precision_at_recall(y_val, p_simple):.4f}")


sample_weight_tr = compute_sample_weight(class_weight="balanced", y=y_tr)

valid threshold: 2026-04-16 00:00:00, train: 8400, valid: 2691

=== Baseline ===
константа (доля ботов): P@R0.7 = 0.0818
RF (2 признака):         P@R0.7 = 0.0920


In [12]:
# ============================================================================
# МОДУЛЬ 3.1 — TIME-SERIES CROSS-VALIDATION
# ============================================================================

# ---------------------------------------------------------------
# Stability check: 3-fold TimeSeriesSplit по времени
# ---------------------------------------------------------------
def time_series_folds(starts, ends, n_splits=3):
    """Фолды по времени с зазором: в train-части фолда только cookie,
    чьё окно закончилось до начала valid-части."""
    # Сортируем по времени, чтобы validation всегда была хронологически позже train.
    order = np.argsort(starts.values)
    fold_size = len(order) // (n_splits + 1)
    folds = []
    for i in range(n_splits):
        train_end = fold_size * (i + 1)
        valid_end = fold_size * (i + 2) if i < n_splits - 1 else len(order)
        va_idx = order[train_end:valid_end]
        tr_idx = order[:train_end]
        valid_start = starts.values[va_idx].min()
        # Убираем train-окна, пересекающиеся с validation: это временной gap.
        tr_idx = tr_idx[ends.values[tr_idx] <= valid_start]
        if len(tr_idx) == 0:      # весь train-фрагмент перекрывается с valid
            continue
        folds.append((tr_idx, va_idx))
    return folds


def evaluate_stability_lgbm(X, y, feature_cols, cat_cols, folds, seeds=(0, 1, 2)):
    results = []
    for fi, (tr_idx, va_idx) in enumerate(folds):
        y_tr_f, y_va_f = y[tr_idx], y[va_idx]
        if y_va_f.sum() < 30:
            print(f"[fold {fi}] слишком мало положительных ({y_va_f.sum()}), пропускаем")
            continue
        sw = compute_sample_weight(class_weight="balanced", y=y_tr_f)
        fold_scores = []
        for seed in seeds:
            m = LGBMClassifier(
                n_estimators=500, learning_rate=0.03, num_leaves=31,
                min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
                random_state=seed, verbosity=-1,
            )
            m.fit(X.iloc[tr_idx][feature_cols], y_tr_f,
                  sample_weight=sw, categorical_feature=cat_cols)
            p = m.predict_proba(X.iloc[va_idx][feature_cols])[:, 1]
            fold_scores.append(precision_at_recall(y_va_f, p))
        results.append({
            "fold": fi,
            "n_train": len(tr_idx), "n_valid": len(va_idx),
            "n_pos_valid": int(y_va_f.sum()),
            "mean": np.mean(fold_scores), "std": np.std(fold_scores),
            "min": np.min(fold_scores), "max": np.max(fold_scores),
        })
    return pd.DataFrame(results)


folds = time_series_folds(train["window_start_ts"], train["window_end_ts"])
stability = evaluate_stability_lgbm(
    Xtr, y, feature_cols, cat_cols, folds, seeds=(0, 1, 2))

print("=== Stability across folds (LGBM, 3 seeds each) ===")
print(stability.round(4))
print(f"\nFold mean P@R0.7:  {stability['mean'].mean():.4f}")
print(f"Fold std:          {stability['mean'].std():.4f}")
print(f"Range:             [{stability['mean'].min():.4f}, {stability['mean'].max():.4f}]")

=== Stability across folds (LGBM, 3 seeds each) ===
   fold  n_train  n_valid  n_pos_valid    mean     std     min     max
0     0     2403     2772          235  0.7155  0.0126  0.7051  0.7333
1     1     5113     2772          209  0.6527  0.0111  0.6422  0.6682
2     2     7599     2775          228  0.7010  0.0144  0.6867  0.7207

Fold mean P@R0.7:  0.6898
Fold std:          0.0329
Range:             [0.6527, 0.7155]


In [13]:
# ============================================================================
# МОДУЛЬ 3.2 — БАЗОВЫЙ LIGHTGBM
# ============================================================================

lgbm = LGBMClassifier(
    n_estimators=700,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=15,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
lgbm.fit(
    Xtr.loc[is_train, feature_cols],
    y_tr,
    sample_weight=sample_weight_tr,
    categorical_feature=cat_cols,
)
p_val_lgbm = lgbm.predict_proba(Xtr.loc[is_valid, feature_cols])[:, 1]

print(f"LightGBM:  P@R0.7 = {precision_at_recall(y_val, p_val_lgbm):.4f}")
print(f"  ROC-AUC = {roc_auc_score(y_val, p_val_lgbm):.4f}")
print(f"  PR-AUC  = {average_precision_score(y_val, p_val_lgbm):.4f}")

imp_lgbm = pd.Series(lgbm.booster_.feature_importance("gain"), index=feature_cols)
print("\nТоп-20 по gain:")
print(imp_lgbm.sort_values(ascending=False).head(20))

LightGBM:  P@R0.7 = 0.7064
  ROC-AUC = 0.9289
  PR-AUC  = 0.7512

Топ-20 по gain:
median_interval_s            28952.795937
pointer_x_std                16887.841319
p10_interval_s                9797.552816
category_nunique              8966.531429
pointer_jump_mean             8320.281187
location_nunique              8139.250664
iqr_interval_s                6487.429036
max_events_per_min            5051.371582
item_nunique                  4976.063762
pointer_dist_std              4704.483620
max_search_page               4295.343341
cookie_age_hours              4252.498129
min_interval_s                3811.980027
frac_interval_near_median     3099.278451
autocorr_lag1                 2954.950425
ratio_photo_swipe             2844.628438
log_interval_std              2699.207607
sync_rel_max_60s              2603.491844
views_per_unique_item         2537.780991
sync_mean_60s                 2438.782322
dtype: float64


In [14]:
# ============================================================================
# МОДУЛЬ 3.3 — БАЗОВЫЙ XGBOOST
# ============================================================================

xgb = XGBClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    tree_method="hist",
    enable_categorical=True,
)
xgb.fit(Xtr.loc[is_train, feature_cols], y_tr, sample_weight=sample_weight_tr)
p_val_xgb = xgb.predict_proba(Xtr.loc[is_valid, feature_cols])[:, 1]

print(f"XGBoost:   P@R0.7 = {precision_at_recall(y_val, p_val_xgb):.4f}")

XGBoost:   P@R0.7 = 0.6844


In [15]:
# ============================================================================
# МОДУЛЬ 3.4 — ABLATION STUDY
# ============================================================================

def ablate(cols_to_drop, label):
    # Ablation: убираем целую группу признаков и измеряем потерю P@R0.7.
    # Так проверяется полезность группы, а не только отдельных колонок.
    cols = [c for c in feature_cols if c not in cols_to_drop]
    m = LGBMClassifier(
        n_estimators=500, learning_rate=0.03, num_leaves=31,
        min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbosity=-1,
    )
    m.fit(Xtr.loc[is_train, cols], y_tr,
          sample_weight=sample_weight_tr,
          categorical_feature=[c for c in cat_cols if c in cols])
    p = m.predict_proba(Xtr.loc[is_valid, cols])[:, 1]
    score = precision_at_recall(y_val, p)
    print(f"{label:40s} P@R0.7 = {score:.4f}")
    return score


SYNC_COLS = ["sync_max_1s", "sync_mean_1s", "sync_frac_1s", "sync_max_60s",
             "sync_mean_60s", "sync_rel_max_60s"]
ITEM_SYNC_COLS = ["item_sync_max", "item_sync_mean", "item_sync_frac_ge2"]
COMPOSITE_COLS = ["irm_combo", "intensity", "regularity", "monotony",
                  "sync_x_cv", "sync_x_regularity"]

print("=== Ablation (LightGBM) ===")
full = ablate([], "baseline (все признаки)")
ablate(SYNC_COLS, "без sync (причинный)")
ablate(ITEM_SYNC_COLS, "без item-sync")
ablate(SYNC_COLS + ITEM_SYNC_COLS, "без ВСЕХ межкукных признаков")
ablate(["cv_interval", "mean_interval_s", "std_interval_s", "median_interval_s",
        "frac_interval_lt1s", "autocorr_lag1", "autocorr_lag2"], "без timing-фичей")
ablate(COMPOSITE_COLS, "без композитов")
ablate(["app_lib_ua_flag"], "без app_lib_ua_flag")
ablate(["cookie_age_hours", "cookie_age_end_hours"], "без возраста cookie")


=== Ablation (LightGBM) ===
baseline (все признаки)                  P@R0.7 = 0.7163
без sync (причинный)                     P@R0.7 = 0.6725
без item-sync                            P@R0.7 = 0.7163
без ВСЕХ межкукных признаков             P@R0.7 = 0.6725
без timing-фичей                         P@R0.7 = 0.6352
без композитов                           P@R0.7 = 0.6569
без app_lib_ua_flag                      P@R0.7 = 0.7163
без возраста cookie                      P@R0.7 = 0.6581


0.6581196581196581

In [16]:
# ============================================================================
# МОДУЛЬ 3.5 — PERMUTATION IMPORTANCE
# ============================================================================

from sklearn.inspection import permutation_importance

perm = permutation_importance(
    lgbm,
    Xtr.loc[is_valid, feature_cols].copy(),
    y_val,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
perm_imp = (pd.Series(perm.importances_mean, index=feature_cols)
            .sort_values(ascending=False))
print("Топ-25 по permutation importance:")
print(perm_imp.head(25).round(4))

Топ-25 по permutation importance:
median_interval_s            0.0725
category_nunique             0.0562
pointer_x_std                0.0502
location_nunique             0.0422
pointer_jump_mean            0.0411
iqr_interval_s               0.0281
p10_interval_s               0.0241
ratio_photo_swipe            0.0183
views_per_unique_item        0.0157
min_interval_s               0.0142
contact_to_view_ratio        0.0136
max_search_page              0.0123
max_events_per_min           0.0115
item_nunique                 0.0115
ua_os                        0.0109
favorite_to_view_ratio       0.0084
ratio_favorite_add           0.0059
frac_interval_near_median    0.0058
query_nunique                0.0055
log_interval_std             0.0049
pointer_y_mean               0.0037
ratio_contact_phone_show     0.0036
n_events                     0.0034
pointer_dist_std             0.0033
cnt_photo_swipe              0.0033
dtype: float64


## 4. Гиперпараметрическая оптимизация

После baseline и проверки признаков подбираем параметры LightGBM по временной кросс-валидации. Целевая функция — `P@R0.7`, то есть оптимизируется именно метрика соревнования.

In [ ]:
# ============================================================================
# МОДУЛЬ 4.1 — OPTUNA: ОПТИМИЗАЦИЯ LIGHTGBM
# ============================================================================

def lgbm_cv_score(params, folds):
    scores = []
    for tr_idx, va_idx in folds:
        y_tr_f, y_va_f = y[tr_idx], y[va_idx]
        if y_va_f.sum() < 20:
            continue
        sw = compute_sample_weight(class_weight="balanced", y=y_tr_f)
        model = LGBMClassifier(**params, random_state=RANDOM_STATE, verbosity=-1)
        model.fit(
            Xtr.iloc[tr_idx][feature_cols], y_tr_f,
            sample_weight=sw,
            categorical_feature=cat_cols,
        )
        p = model.predict_proba(Xtr.iloc[va_idx][feature_cols])[:, 1]
        scores.append(precision_at_recall(y_va_f, p))
    return float(np.mean(scores)) if scores else 0.0


def objective_lgbm(trial):
    # Optuna подбирает параметры по временным фолдам.
    # Цель — непосредственно соревновательная метрика P@R0.7.
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 1e-3, 1.0, log=True),
    }
    return lgbm_cv_score(params, folds)


# study_lgbm = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
# study_lgbm.optimize(objective_lgbm, n_trials=130, show_progress_bar=True)

# print("Лучший CV P@R0.7 (LightGBM):", study_lgbm.best_value)
# print("Лучшие параметры:", study_lgbm.best_params)

# Воспроизвести подбор гиперпараметров optuna сложно воспроизвести,
# поэтому тут я оставлю найденные гиперпараметры как словарь

best_value_lgbm = 0.7153679925932392
best_params_lgbm = {'n_estimators': 900, 'learning_rate': 0.019857397148539027, 'num_leaves': 93, 'min_child_samples': 49, 'subsample': 0.7374109982451702, 'colsample_bytree': 0.6274945950060813, 'reg_alpha': 0.0012235371092392201, 'reg_lambda': 0.0020041154751235684, 'min_split_gain': 0.019455167924425233}



print("Лучший CV P@R0.7 (LightGBM):", best_value_lgbm)
print("Лучшие параметры:", best_params_lgbm)

Best trial: 127. Best value: 0.715368: 100%|██████████| 130/130 [09:29<00:00,  4.38s/it]

Лучший CV P@R0.7 (LightGBM): 0.7153679925932392
Лучшие параметры: {'n_estimators': 900, 'learning_rate': 0.019857397148539027, 'num_leaves': 93, 'min_child_samples': 49, 'subsample': 0.7374109982451702, 'colsample_bytree': 0.6274945950060813, 'reg_alpha': 0.0012235371092392201, 'reg_lambda': 0.0020041154751235684, 'min_split_gain': 0.019455167924425233}


In [ ]:
# ============================================================================
# МОДУЛЬ 4.2 — ФИНАЛЬНЫЙ LIGHTGBM
# ============================================================================

# lgbm_params_best = dict(study_lgbm.best_params)
lgbm_params_best = best_params_lgbm
lgbm_params_best.update({"random_state": RANDOM_STATE, "verbosity": -1})

print("LGBM params из Optuna:")
# for k, v in study_lgbm.best_params.items():
for k, v in best_params_lgbm.items():
    print(f"  {k:20s} = {v}")
# print(f"  CV P@R0.7 (3 folds) = {study_lgbm.best_value:.4f}\n")
print(f"  CV P@R0.7 (3 folds) = {best_value_lgbm:.4f}\n")

lgbm = LGBMClassifier(**lgbm_params_best)
lgbm.fit(
    Xtr.loc[is_train, feature_cols], y_tr,
    sample_weight=sample_weight_tr,
    categorical_feature=cat_cols,
)
p_val_lgbm = lgbm.predict_proba(Xtr.loc[is_valid, feature_cols])[:, 1]
lgbm_val_score = precision_at_recall(y_val, p_val_lgbm)

print(f"LightGBM (val):  P@R0.7 = {lgbm_val_score:.4f}")
print(f"  ROC-AUC = {roc_auc_score(y_val, p_val_lgbm):.4f}")
print(f"  PR-AUC  = {average_precision_score(y_val, p_val_lgbm):.4f}")

LGBM params из Optuna:
  n_estimators         = 900
  learning_rate        = 0.019857397148539027
  num_leaves           = 93
  min_child_samples    = 49
  subsample            = 0.7374109982451702
  colsample_bytree     = 0.6274945950060813
  reg_alpha            = 0.0012235371092392201
  reg_lambda           = 0.0020041154751235684
  min_split_gain       = 0.019455167924425233
  CV P@R0.7 (3 folds) = 0.7154

LightGBM (val):  P@R0.7 = 0.7009
  ROC-AUC = 0.9307
  PR-AUC  = 0.7516


In [ ]:
# ============================================================================
# МОДУЛЬ 4.3 — ЗАФИКСИРОВАННЫЕ РЕЗУЛЬТАТЫ LIGHTGBM
# ============================================================================

# LGBM params из Optuna:
#   n_estimators         = 900
#   learning_rate        = 0.019857397148539027
#   num_leaves           = 93
#   min_child_samples    = 49
#   subsample            = 0.7374109982451702
#   colsample_bytree     = 0.6274945950060813
#   reg_alpha            = 0.0012235371092392201
#   reg_lambda           = 0.0020041154751235684
#   min_split_gain       = 0.019455167924425233
#   CV P@R0.7 (3 folds) = 0.7154

# LightGBM (val):  P@R0.7 = 0.7009
#   ROC-AUC = 0.9307
#   PR-AUC  = 0.7516

## 5. Оптимизация XGBoost

XGBoost настраивается по той же временной схеме, поэтому результаты двух моделей можно сравнивать напрямую.

In [ ]:
# ============================================================================
# МОДУЛЬ 5.1 — OPTUNA: ОПТИМИЗАЦИЯ XGBOOST
# ============================================================================


folds = time_series_folds(train["window_start_ts"], train["window_end_ts"], n_splits=3)

def xgb_cv_score(params, folds):
    scores = []
    for tr_idx, va_idx in folds:
        y_tr_f, y_va_f = y[tr_idx], y[va_idx]
        if y_va_f.sum() < 20:
            continue
        sw = compute_sample_weight(class_weight="balanced", y=y_tr_f)
        model = XGBClassifier(
            **params, random_state=RANDOM_STATE,
            eval_metric="logloss", tree_method="hist", enable_categorical=True,
        )
        model.fit(Xtr.iloc[tr_idx][feature_cols], y_tr_f, sample_weight=sw)
        p = model.predict_proba(Xtr.iloc[va_idx][feature_cols])[:, 1]
        scores.append(precision_at_recall(y_va_f, p))
    return float(np.mean(scores)) if scores else 0.0


def objective(trial):
    # Для XGBoost используется та же временная схема, что и для LightGBM.
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 10, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-3, 2.0, log=True),
    }
    return xgb_cv_score(params, folds)


# study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
# study.optimize(objective, n_trials=130, show_progress_bar=True)

# print("Лучший CV P@R0.7:", study.best_value)
# print("Лучшие параметры:", study.best_params)

best_value = 0.707616968949735
best_params = {'n_estimators': 850, 'learning_rate': 0.010008370186059636, 'max_depth': 8, 'min_child_weight': 7.135987686944493, 'subsample': 0.6978724116204501, 'colsample_bytree': 0.5708146533796771, 'reg_alpha': 0.007541875042090004, 'reg_lambda': 0.031486299114833895, 'gamma': 0.009131503613285026}


print("Лучший CV P@R0.7 (LightGBM):", best_value)
print("Лучшие параметры:", best_params)

Best trial: 63. Best value: 0.707617: 100%|██████████| 130/130 [17:36<00:00,  8.13s/it]

Лучший CV P@R0.7: 0.707616968949735
Лучшие параметры: {'n_estimators': 850, 'learning_rate': 0.010008370186059636, 'max_depth': 8, 'min_child_weight': 7.135987686944493, 'subsample': 0.6978724116204501, 'colsample_bytree': 0.5708146533796771, 'reg_alpha': 0.007541875042090004, 'reg_lambda': 0.031486299114833895, 'gamma': 0.009131503613285026}


In [ ]:
# ============================================================================
# МОДУЛЬ 5.2 — ФИНАЛЬНЫЙ XGBOOST
# ============================================================================

# xgb_params_best = dict(study.best_params)
xgb_params_best = best_params
xgb_params_best.update({
    "random_state": RANDOM_STATE,
    "eval_metric": "logloss",
    "tree_method": "hist",
    "enable_categorical": True,
})

print("XGB params из Optuna:")
# for k, v in study.best_params.items():
for k, v in best_params.items():
    print(f"  {k:20s} = {v}")
# print(f"  CV P@R0.7 (3 folds) = {study.best_value:.4f}\n")
print(f"  CV P@R0.7 (3 folds) = {best_value:.4f}\n")

xgb = XGBClassifier(**xgb_params_best)
xgb.fit(Xtr.loc[is_train, feature_cols], y_tr, sample_weight=sample_weight_tr)
p_val_xgb = xgb.predict_proba(Xtr.loc[is_valid, feature_cols])[:, 1]
xgb_val_score = precision_at_recall(y_val, p_val_xgb)

print(f"XGBoost (val):  P@R0.7 = {xgb_val_score:.4f}")
print(f"  ROC-AUC = {roc_auc_score(y_val, p_val_xgb):.4f}")
print(f"  PR-AUC  = {average_precision_score(y_val, p_val_xgb):.4f}")

XGB params из Optuna:
  n_estimators         = 850
  learning_rate        = 0.010008370186059636
  max_depth            = 8
  min_child_weight     = 7.135987686944493
  subsample            = 0.6978724116204501
  colsample_bytree     = 0.5708146533796771
  reg_alpha            = 0.007541875042090004
  reg_lambda           = 0.031486299114833895
  gamma                = 0.009131503613285026
  CV P@R0.7 (3 folds) = 0.7076

XGBoost (val):  P@R0.7 = 0.7091
  ROC-AUC = 0.9315
  PR-AUC  = 0.7519


In [ ]:
# ============================================================================
# МОДУЛЬ 5.3 — ЗАФИКСИРОВАННЫЕ РЕЗУЛЬТАТЫ XGBOOST
# ============================================================================

# XGB params из Optuna:
#   n_estimators         = 850
#   learning_rate        = 0.010008370186059636
#   max_depth            = 8
#   min_child_weight     = 7.135987686944493
#   subsample            = 0.6978724116204501
#   colsample_bytree     = 0.5708146533796771
#   reg_alpha            = 0.007541875042090004
#   reg_lambda           = 0.031486299114833895
#   gamma                = 0.009131503613285026
#   CV P@R0.7 (3 folds) = 0.7076

# XGBoost (val):  P@R0.7 = 0.7091
#   ROC-AUC = 0.9315
#   PR-AUC  = 0.7519

## 6. Ансамблирование

LightGBM и XGBoost могут ошибаться на разных cookie. Перебираем вес `w` и используем простое взвешенное среднее их вероятностей.

In [ ]:
# ============================================================================
# МОДУЛЬ 6 — АНСАМБЛЬ LIGHTGBM + XGBOOST
# ============================================================================

# Поиск веса ансамбля по валидации
print("\n=== Поиск веса ансамбля ===")
best_w, best_score = 0.5, 0.0
# w — доля LightGBM; (1-w) — доля XGBoost.
# Проверяем простое взвешенное среднее вероятностей.
for w in np.arange(0.0, 1.01, 0.1):
    p = w * p_val_lgbm + (1 - w) * p_val_xgb
    s = precision_at_recall(y_val, p)
    marker = ""
    if s > best_score:
        best_score, best_w = s, w
        marker = "  <-- лучше"
    print(f"  w_lgbm={w:.1f}  P@R0.7 = {s:.4f}{marker}")

print(f"\nЛучший вес: w_lgbm={best_w:.1f}, P@R0.7 = {best_score:.4f}")

p_val_ens = best_w * p_val_lgbm + (1 - best_w) * p_val_xgb
print(f"Ансамбль:  P@R0.7 = {precision_at_recall(y_val, p_val_ens):.4f}")

print("\n=== Итоговая таблица на валидации ===")
rows = [
    ("константа",            y_val.mean()),
    ("RF (2 признака)",      precision_at_recall(y_val, p_simple)),
    ("LightGBM",             precision_at_recall(y_val, p_val_lgbm)),
    ("XGBoost",              precision_at_recall(y_val, p_val_xgb)),
    (f"Ансамбль (w={best_w:.1f})", precision_at_recall(y_val, p_val_ens)),
]
for name, v in rows:
    print(f"  {name:22s} {v:.4f}")


=== Поиск веса ансамбля ===
  w_lgbm=0.0  P@R0.7 = 0.7091  <-- лучше
  w_lgbm=0.1  P@R0.7 = 0.7097  <-- лучше
  w_lgbm=0.2  P@R0.7 = 0.7045
  w_lgbm=0.3  P@R0.7 = 0.7045
  w_lgbm=0.4  P@R0.7 = 0.7097
  w_lgbm=0.5  P@R0.7 = 0.7032
  w_lgbm=0.6  P@R0.7 = 0.7059
  w_lgbm=0.7  P@R0.7 = 0.7130  <-- лучше
  w_lgbm=0.8  P@R0.7 = 0.7097
  w_lgbm=0.9  P@R0.7 = 0.7064
  w_lgbm=1.0  P@R0.7 = 0.7009

Лучший вес: w_lgbm=0.7, P@R0.7 = 0.7130
Ансамбль:  P@R0.7 = 0.7130

=== Итоговая таблица на валидации ===
  константа              0.0818
  RF (2 признака)        0.0920
  LightGBM               0.7009
  XGBoost                0.7091
  Ансамбль (w=0.7)       0.7130


## 7. Финальная модель и submission

После выбора признаков, гиперпараметров и веса ансамбля обе модели обучаются на полном train. Затем строится `submission.csv` для test.

In [ ]:
# ============================================================================
# МОДУЛЬ 7 — ФИНАЛЬНОЕ ОБУЧЕНИЕ И SUBMISSION
# ============================================================================

# После выбора архитектуры переобучаем обе модели на всём train.
# Валидация уже использована для выбора параметров и веса ансамбля.
sample_weight_full = compute_sample_weight(class_weight="balanced", y=y)

lgbm_final = LGBMClassifier(**lgbm_params_best)
lgbm_final.fit(          # обучаем на всем train
    Xtr[feature_cols], y,
    sample_weight=sample_weight_full,
    categorical_feature=cat_cols,
)

xgb_final = XGBClassifier(**xgb_params_best)
xgb_final.fit(Xtr[feature_cols], y, sample_weight=sample_weight_full)

p_lgbm = lgbm_final.predict_proba(Xte[feature_cols])[:, 1]
p_xgb  = xgb_final.predict_proba(Xte[feature_cols])[:, 1]
scores = best_w * p_lgbm + (1 - best_w) * p_xgb   # вес с валидации

submission = pd.DataFrame({"cookie_id": Xte["cookie_id"], "score": scores})

assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert submission["score"].between(0, 1).all()

submission.to_csv("submission.csv", index=False)
print(f"Сохранено: submission.csv ({len(submission)} строк)")
submission.head()

Сохранено: submission.csv (4909 строк)


,cookie_id,score
0,ck_315fb710a0e371e7,0.001271
1,ck_a76ee3b3e3e522fd,0.084415
2,ck_94c9a4d382689e82,0.042013
3,ck_8eaf9509ad9462a0,0.001586
4,ck_9a88a5a989cb5bc6,0.009330
